In [1]:
import os
os.getcwd()

'c:\\Users\\Usuario\\OneDrive - UNIVERSIDAD DE HUELVA\\Granada\\TrabajoFM\\scripts\\Python_Pipeline_SWAT_Pascal\\swat_pipeline\\trabajoFM\\python_pipeline_scripts\\script POINT loads - input .dat'

## GIS part

### Non ArcPy version

In [2]:
#!pip install geopandas rasterio rasterstats shapely fiona numpy matplotlib


In [3]:
import os
import glob
import rasterio
import rasterio.mask
import geopandas as gpd
from rasterstats import zonal_stats
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.geometry import box
import matplotlib.pyplot as plt
import numpy as np

def reproject_raster_to_epsg(input_path, output_path, epsg=25830, overwrite_cache=False):
    if os.path.exists(output_path) and not overwrite_cache:
        print(f"⚠ Skipping reprojection: {output_path} already exists (cached)")
        return

    print(f"→ Reprojecting raster {input_path} to EPSG:{epsg}")
    with rasterio.open(input_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, f"EPSG:{epsg}", src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": f"EPSG:{epsg}",
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(output_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=f"EPSG:{epsg}",
                    resampling=Resampling.nearest
                )
    print(f"✔ Reprojected raster saved to {output_path}")


def clip_raster_with_bounds(input_path, output_path, bounds, overwrite_cache=False):
    if os.path.exists(output_path) and not overwrite_cache:
        print(f"⚠ Skipping clipping: {output_path} already exists (cached)")
        return

    print(f"→ Clipping raster {input_path} to bounds {bounds}")
    geom = [box(*bounds)]
    with rasterio.open(input_path) as src:
        out_image, out_transform = rasterio.mask.mask(src, geom, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(out_image)
    print(f"✔ Clipped raster saved to {output_path}")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import geopandas as gpd
from matplotlib.colors import Normalize, LinearSegmentedColormap

def plot_raster_and_zones(raster_path, gdf, title):
    with rasterio.open(raster_path) as src:
        fig, ax = plt.subplots(figsize=(10, 10), facecolor='white')
        raster_data = src.read(1).astype(float)

        nodata = src.nodata
        # Mask nodata and zero values as NaN (transparent)
        if nodata is not None:
            raster_data[(raster_data == nodata) | (raster_data == 0)] = np.nan
        else:
            raster_data[raster_data == 0] = np.nan

        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

        # Find min positive value to avoid black on black for smallest values
        min_val = np.nanmin(raster_data)
        max_val = np.nanmax(raster_data)

        # Define a simple grey colormap from light grey to black
        colors = [(0.9, 0.9, 0.9), (0.0, 0.0, 0.0)]  # light grey to black
        cmap = LinearSegmentedColormap.from_list('lightgrey_to_black', colors)

        # Normalize with min and max
        norm = Normalize(vmin=min_val, vmax=max_val)

        im = ax.imshow(
            raster_data,
            extent=extent,
            origin='upper',
            cmap=cmap,
            norm=norm,
            alpha=1.0
        )

        # Plot zone boundaries with black edges
        gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=1)

        # Add GRIDCODE labels
        for _, row in gdf.iterrows():
            centroid = row.geometry.centroid
            ax.text(
                centroid.x, centroid.y,
                str(row["GRIDCODE"]),
                fontsize=9,
                fontweight='bold',
                color='red',
                ha='center',
                va='center',
                bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2', alpha=0.7)
            )

        ax.set_facecolor('white')
        ax.set_title(title, color='black')
        ax.set_xlabel("Easting", color='black')
        ax.set_ylabel("Northing", color='black')
        ax.tick_params(colors='black')

        cbar = plt.colorbar(im, ax=ax, shrink=0.7)
        cbar.set_label('Population', color='black')
        cbar.ax.yaxis.set_tick_params(color='black')
        plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='black')

        plt.grid(False)
        plt.tight_layout()
        plt.show()



def subbasinPopulationAggregationToGPKG(raster_folder, sub_basin_fp, zone_field, output_gpkg, overwrite_cache=False):
    print(f"→ Reading sub-basin shapefile: {sub_basin_fp}")
    gdf = gpd.read_file(sub_basin_fp).to_crs("EPSG:25830")

    bounds = gdf.total_bounds
    print(f"✔ Reprojected sub-basins to EPSG:25830")
    print(f"→ Using bounding box for clipping: {bounds}")

    tif_files = glob.glob(os.path.join(raster_folder, "hipgdac_es_100_*.tif"))

    # Create cache directory
    cache_dir = os.path.join(raster_folder, "temp_cache")
    os.makedirs(cache_dir, exist_ok=True)

    for raster_path in tif_files:
        raster_filename = os.path.splitext(os.path.basename(raster_path))[0]
        year = raster_filename.split("_")[-1]
        field_name = f"popul_sum_{year}"
        print(f"\n🔹 Processing raster: {raster_path} → year: {year}")

        proj_raster_path = os.path.join(cache_dir, f"{raster_filename}_reproj.tif")
        clip_raster_path = os.path.join(cache_dir, f"{raster_filename}_clip.tif")

        # Step 1: Reproject
        reproject_raster_to_epsg(raster_path, proj_raster_path, epsg=25830, overwrite_cache=overwrite_cache)

        # Step 2: Stats before clipping
        with rasterio.open(proj_raster_path) as src:
            data = src.read(1).astype(np.float32)
            data[data == 0] = np.nan
            print(f"→ Raster stats BEFORE clip:")
            print(f"   MIN: {np.nanmin(data)}, MAX: {np.nanmax(data)}, MEAN: {np.nanmean(data)}, SUM: {np.nansum(data)}")

        # Step 3: Clip
        clip_raster_with_bounds(proj_raster_path, clip_raster_path, bounds, overwrite_cache=overwrite_cache)

        # Step 4: Stats after clipping
        with rasterio.open(clip_raster_path) as src:
            clipped = src.read(1).astype(np.float32)
            clipped[clipped == 0] = np.nan
            print(f"→ Raster stats AFTER clip:")
            print(f"   MIN: {np.nanmin(clipped)}, MAX: {np.nanmax(clipped)}, MEAN: {np.nanmean(clipped)}, SUM: {np.nansum(clipped)}")

        # Step 5: Plot
        print("→ Plotting raster with sub-basin overlay:")
        plot_raster_and_zones(clip_raster_path, gdf, title=f"Raster {year} and Sub-basin Alignment")

        # Step 6: Zonal stats
        with rasterio.open(clip_raster_path) as src:
            nodata_val = 0
        stats = zonal_stats(
            gdf,
            clip_raster_path,
            stats=["sum", "count"],
            geojson_out=False,
            nodata=nodata_val,
            all_touched=False
        )

        sums = [round(s["sum"]) if s["sum"] is not None else 0 for s in stats]
        counts = [s["count"] if s["count"] is not None else 0 for s in stats]

        gdf[field_name] = np.array(sums, dtype=np.float64)
        gdf[f"pixels_included_{year}"] = np.array(counts, dtype=np.int32)

        print(f"✔ Zonal statistics for {field_name}:")
        for idx, stat in enumerate(stats):
            zone_id = gdf.iloc[idx][zone_field]
            s = stat["sum"] or 0
            c = stat["count"] or 0
            avg = s / c if c else "NA"
            print(f"   Zone {zone_id}: SUM={s}, COUNT={c}, AVG={avg}")

        # Step 7: Debug Zone 10
        debug_zone_index = 9
        zone_geom = [gdf.iloc[debug_zone_index].geometry]
        with rasterio.open(clip_raster_path) as src:
            out_image, _ = rasterio.mask.mask(src, zone_geom, crop=True)
            values = out_image[0].flatten()
            valid_values = values[values > 0]
            print(f"🔍 Debug: Zone 10 pixel values (non-zero): {valid_values[:10]}...")
            print(f"→ Zone 10 pixel count (non-zero): {len(valid_values)}, sum: {np.sum(valid_values)}")

        # Step 8: Cleanup
        if overwrite_cache:
            print("→ Cleaning up intermediate files")
            for f in [proj_raster_path, clip_raster_path]:
                if os.path.exists(f):
                    os.remove(f)
        else:
            print("⚠ Keeping intermediate files for future runs (cached)")

    output_layer_name = "population_by_subbasin"
    print(f"\n→ Writing results to GeoPackage: {output_gpkg} (layer: {output_layer_name})")



    # Separate column lists
    pixel_cols = sorted([col for col in gdf.columns if col.startswith("pixels_included_")])
    popul_cols = sorted([col for col in gdf.columns if col.startswith("popul_sum_")])

    # Keep all other columns (non these two types)
    other_cols = [col for col in gdf.columns if col not in pixel_cols + popul_cols]

    # New column order: other columns + pixels_included fields + popul_sum fields
    new_col_order = other_cols + pixel_cols + popul_cols

    # Reorder GeoDataFrame columns
    gdf = gdf[new_col_order]

    # Then save

    
    gdf.to_file(output_gpkg, layer=output_layer_name, driver="GPKG")
    gdf.to_csv("output.csv", float_format="%.8f", index=False)

    print("✔ Done")

    return os.path.join(output_gpkg, output_layer_name)


In [6]:
import os
import pandas as pd
from pathlib import Path

# Define paths
raster_folder = r"..\..\..\..\..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\fgoerlich-HIPGDAC-ES-cd11f21\HIPGDAC-ES\1970-2021 copy"
basin_fc = r"..\..\..\..\..\..\Genil GEO_INFO_POOL\SWaT outputs\Cubillas\shapes cubillas\Sub_basin.shp"
output_folder = r"..\..\..\..\..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source"
output_gpkg = os.path.join(output_folder, "cubillas_population.gpkg")

# Make sure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Run processing function (assumes you already defined the open-source version above)
result_layer_path = subbasinPopulationAggregationToGPKG(
    raster_folder=raster_folder,
    sub_basin_fp=basin_fc,
    zone_field="GRIDCODE",
    output_gpkg=output_gpkg,
    overwrite_cache=False  # Set True to force reprocessing
)

# Read the result layer from the GeoPackage
layer_name = "population_by_subbasin"
gdf = gpd.read_file(output_gpkg, layer=layer_name)

# Export to CSV (excluding geometry)
csv_output = output_gpkg.replace(".gpkg", ".csv")
gdf.drop(columns="geometry").to_csv(csv_output, sep=';', index=False)

print(f"Exported to CSV: {csv_output}")


→ Reading sub-basin shapefile: ..\..\..\..\..\..\Genil GEO_INFO_POOL\SWaT outputs\Cubillas\shapes cubillas\Sub_basin.shp
✔ Reprojected sub-basins to EPSG:25830
→ Using bounding box for clipping: [ 438912.5        4123462.50012207  470812.5        4158362.50012207]

→ Writing results to GeoPackage: ..\..\..\..\..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.gpkg (layer: population_by_subbasin)
✔ Done
Exported to CSV: ..\..\..\..\..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.csv


# Non-GIS part

## Define non GIS script's inputs & outputs

In [8]:
import pandas as pd
import os
os.getcwd()

#### Load the CSV file (relative path to this scripts folder)
# The CSV file should have the last columns representing years and their population counts and the first column being an Unique ID
# optimally this CSV comes out of the ArcGIS Pro Model in T"rabajoFM\Genil_ArcGIS_Pascal"

basin_name = "Cubillas"
# Population aggregated according to (sub)basins' subbasins:
file_path = r'..\..\..\..\..\..\Genil GEO_INFO_POOL\Input Data\Population data\Basin Aggregations\Cubillas population loads\cuenca_cubillas_habitantes_decadas_1970_2021_arcgis_output.csv'
file_path_2 = r"..\..\..\..\..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.csv"
identifier_column = "GRIDCODE"

# to extract years that have been and shall be simulated
cio_file = r"C:\SWAT\ArcSWAT\Databases\cubillas_hru_playground\Scenarios\Default\TxtInOut\file.cio"

output_dir = r".\swat_ready_recyear_files"


df = pd.read_csv(file_path_2, sep=';')
df.head()

,OBJECTID,GRIDCODE,Subbasin,Area,Slo1,Len1,Sll,Csl,Wid1,Dep1,Lat,Long_,Elev,ElevMin,ElevMax,Bname,Shape_Leng,Shape_Area,HydroID,OutletID
0,1,1,1,9411.1875,14.374060,18351.119215,24.382810,4.267118,19.714073,0.800573,37.527300,-3.501996,1098.095473,889.0,1687.0,NaN,62800.0,94111875.0,300001,100016
1,2,2,2,2769.3750,16.043022,10306.701833,18.287108,5.766811,9.462781,0.490790,37.454030,-3.578132,1016.054653,832.0,1496.0,NaN,38800.0,27693750.0,300002,100015
2,3,3,3,5428.3125,16.590860,12456.854250,18.287108,3.447740,14.170612,0.642406,37.463467,-3.381741,1086.639640,889.0,1343.0,NaN,43350.0,54283125.0,300003,100014
3,4,4,4,2955.4375,16.652315,13936.753237,18.287108,2.913354,9.839268,0.503723,37.465203,-3.462121,976.712828,812.0,1217.0,NaN,37850.0,29554375.0,300004,100012
4,5,5,5,2131.1875,17.413874,14816.930906,18.287108,2.831092,8.086539,0.441969,37.451979,-3.438833,1022.809395,812.0,1240.0,NaN,41850.0,21311875.0,300005,100011


In [ ]:
# Convert the last 6 columns to integers, extracting the first part of the string if necessary
for col in df.columns[-6:]:
    df[col] = df[col].apply(lambda x: int(str(x).split('.')[0]))

# Convert the first column to integers 
df[df.columns[0]] = df[df.columns[0]].apply(lambda x: int(str(x).split(',')[0]))

# Define correct year labels
year_labels = [1970, 1981, 1991, 2001, 2011, 2021]

# Replace the current column names (last 6) with correct years
df.rename(columns=dict(zip(df.columns[-6:], year_labels)), inplace=True)
df.head(20)

## Inter- & extra-polate decade data

In [10]:

def interpolate_dataframe(df, id_column, year_start, year_end, num_year_cols=6):
    """
    Interpolates numeric year data for each unique ID in a DataFrame so that all years that are between existent colums get a column with interpolated values

    Parameters:
    - df: input DataFrame
    - id_column: name of the ID column (string)
    - year_start: start year (inclusive, int)
    - year_end: end year (inclusive, int)
    - num_year_cols: how many columns from the end to treat as year columns (default 6)

    Returns:
    - result_wide: DataFrame with interpolated values, one row per ID, with 'identifier' column first
    """
    # Select year columns (last N columns)
    year_cols = df.columns[-num_year_cols:]
    
    # Clean year columns: drop after comma, convert to int
    for col in year_cols:
        df[col] = df[col].apply(lambda x: int(str(x).split(',')[0]))

    # Create full year range
    full_years = pd.DataFrame({'Year': range(year_start, year_end + 1)})
    
    # Prepare result table
    result = pd.DataFrame({'Year': full_years['Year']})
    
    # Interpolate for each unique ID
    for _, group in df.groupby(id_column):
        subset = group.melt(id_vars=[id_column], value_vars=year_cols, var_name='Year', value_name='Value')
        subset['Year'] = subset['Year'].astype(int)
        merged = full_years.merge(subset, on='Year', how='left').sort_values('Year')
        merged['Value'] = merged['Value'].interpolate(method='linear').ffill().bfill()
        label = f"{group[id_column].values[0]}"
        result[label] = merged['Value'].values

    # Reshape to wide format
    result_wide = result.set_index('Year').T.reset_index()

    # Assign 'identifier' column
    if pd.api.types.is_numeric_dtype(df[id_column]):
        result_wide['identifier'] = pd.to_numeric(result_wide['index'], errors='raise')
    else:
        result_wide['identifier'] = result_wide['index']

    # Drop helper column and reorder
    result_wide.drop(columns=['index'], inplace=True)
    cols = ['identifier'] + [col for col in result_wide.columns if col != 'identifier']
    result_wide = result_wide[cols]

    # Ensure all numeric columns are integers (drop any decimals)
    numeric_cols = result_wide.columns[1:]  # exclude 'identifier'
    result_wide[numeric_cols] = result_wide[numeric_cols].applymap(lambda x: int(float(x)))

    # Rename columns: just the year numbers (no label 'Year')
    result_wide.columns = ['identifier'] + [str(year) for year in range(year_start, year_end + 1)]
    
    return result_wide



In [11]:
def extrapolate_to_2025_with_fill(df):
    """
    Extrapolates numeric trends from the last two decade columns 
    and fills all years up to 2025 with interpolated/extrapolated values.

    Parameters:
    - df: input DataFrame
    - id_column: name of the ID column (string)

    Returns:
    - df_filled: DataFrame with new year columns up to 2025
    """
    # Identify year columns (numeric names only)
    year_cols = [col for col in df.columns if str(col).isdigit()]
    year_cols_sorted = sorted(year_cols, key=int)

    last_year = int(year_cols_sorted[-1])
    decade_earlier = int(year_cols_sorted[-2])

    # Calculate slope per year
    year_diff = last_year - decade_earlier
    new_years = list(range(last_year + 1, 2026))

    # Make a copy
    df_copy = df.copy()

    # Clean numeric values (strip commas, cast to int)
    for col in [str(decade_earlier), str(last_year)]:
        df_copy[col] = df_copy[col].apply(lambda x: int(str(x).split(',')[0]))

    # For each row, compute and fill values for each new year
    for year in new_years:
        df_copy[str(year)] = df_copy.apply(
            lambda row: row[str(last_year)] + ((year - last_year) / year_diff) * (row[str(last_year)] - row[str(decade_earlier)]),
            axis=1
        ).round().astype(int)

    return df_copy


## Building yearly population time series, up till present

In [12]:
df_interpol_1970_to_2021 = interpolate_dataframe(df, id_column=identifier_column, year_start=1970, year_end=2021, num_year_cols=6)

ValueError: invalid literal for int() with base 10: 'nan'

In [13]:
df_1970_to_2025 = extrapolate_to_2025_with_fill(df_interpol_1970_to_2021)
df_1970_to_2025 

NameError: name 'df_interpol_1970_to_2021' is not defined

In [14]:
# Visualize trends

try:
    import matplotlib.pyplot as plt
    ENABLE_PLOTTING = True
except ImportError:
    ENABLE_PLOTTING = False

if ENABLE_PLOTTING:

    plt.figure(figsize=(14, 8))

    series_list = []
    years = [str(y) for y in range(1970, 2025)]

    # Get the name of the first column (used for labels)
    label_column = df_1970_to_2025.columns[0]

    for idx, row in df_1970_to_2025.iterrows():
        values = row[years].values.flatten()
        label = row[label_column]
        first_value = values[0]
        series_list.append((first_value, label, years, values))

    # Sort by first value (descending) so the legend matches the line starting order
    series_list.sort(reverse=True, key=lambda x: x[0])

    # Plot in sorted order
    for _, label, years, values in series_list:
        plt.plot([int(y) for y in years], values, alpha=0.5, linewidth=1, label=label)

    plt.title('Interpolated Population Over Time for All Polygons')
    plt.xlabel('Year')
    plt.ylabel('Population (integer)')
    plt.grid(True)

    # Show legend if manageable
    if len(series_list) <= 15:
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.show()



NameError: name 'df_1970_to_2025' is not defined

<Figure size 1400x800 with 0 Axes>

## Calculating chemical loads from Population data

#### CONSTANTs: assumed wastewater production from per person per day (liters/day) wastwater production & mg/liter concentration values from literature



In [15]:
WASTEWATER_L_PER_PERSON_PER_DAY = 150

150 * 15 * 4600 /1000000

# Concentraciones esperadas (mg/L) en aguas residuales - ORDEN COMO SWAT LO REQUIERE SEGUN ch. 31 del swat 2012 io handbook

# Valores tomado desde Metcalf (2000) - "Ingeniería de aguas residuales: tratamiento, vertido y reutilización"
expected_mgL_values = {
    "ORGNYR": 15,       # Nitrógeno orgánico — proteínas, urea, etc.
    "ORGPYR": 3,        # Fósforo orgánico — asociado a materia particulada
    "NO3YR": 0,         # Nitrato — suele ser 0 en aguas residuales crudas (antes de nitrificación)
    "NH3YR": 25,        # Amoníaco libre — forma principal de N inorgánico en agua residual
    "NO2YR": 0,         # Nitrito — normalmente inestable y cercano a cero
    "MINPYR": 5,        # Fósforo inorgánico soluble (PO₄³⁻) — disponible biológicamente
    "SEDYR": 720,       # Sólidos totales en suspensión — proxy para carga de sedimentos
    "CBODYR": 220,      # Demanda Bioquímica de Oxígeno (CBOD / DBO₅) — carga de materia orgánica biodegradable
    "DISOXYR": 2.5,     # Oxígeno disuelto — suele estar en valores bajos en aguas residuales
    "CHLAYR": 0.001     # Clorofila-a — muy baja en aguas residuales (agua turbia impide crecimiento de algas)
}

###### Comentarios explicativos (referencia para revisión técnica)
# ORGNYR: Organic nitrogen concentration (mg/L) — from proteins, urea, etc.
# ORGPYR: Organic phosphorus concentration (mg/L) — associated with organic matter and detritus
# NO3YR: Nitrate concentration (mg/L) — highly soluble, product of nitrification (usually near zero in raw wastewater)
# NH3YR: Ammonia concentration (mg/L) — reduced nitrogen form, main N species in domestic wastewater
# NO2YR: Nitrite concentration (mg/L) — intermediate in nitrification, usually unstable and near zero
# MINPYR: Mineral (soluble) phosphorus concentration (mg/L) — orthophosphate readily bioavailable
# SEDYR: Suspended solids concentration (mg/L) — total suspended solids proxy, major sediment load
# CBODYR: Carbonaceous BOD (mg/L) — biological oxygen demand (BOD5), high in untreated wastewater
# DISOXYR: Dissolved oxygen (mg/L) — low due to high oxygen consumption
# CHLAYR: Chlorophyll-a (mg/L) — proxy for algae biomass, very low in wastewater due to turbidity and low light

In [16]:
print((150 * 15 * 4600 )/1000000)

10.35


#### Math functions

In [17]:
def mgL_to_kg_day(mg_per_l, persons):
    """
    Convert concentration (mg/L) to total mass per day (kg/day),
    based on wastewater produced per person.
    Formula: mg/L × liters/day × persons ÷ 1,000,000 → kg/day
    """
    return mg_per_l * WASTEWATER_L_PER_PERSON_PER_DAY * persons / 1_000_000

def build_point_load_timeseries_dataframes(row, expected_mgL_values, years, final_columns=None):
    """
    For a given row (representing one unit, e.g., subbasin), 
    build a DataFrame with yearly SWAT point source values from expected mg/L urban wastewater values. 
    OUTPUT: kg/day for each pollutant variable and total wastewater flow (FLOYR) in m³/day.
    Allows specifying the final columns and their order; fills missing columns with 0s.
    """
    df_out = pd.DataFrame({'YEAR': years})
    # Extract population series from the row
    df_out['POPULATION'] = row[[str(y) for y in years]].values.flatten()
    # Calculate total wastewater flow (FLOYR) in m³/day
    df_out['FLOYR'] = df_out['POPULATION'] * WASTEWATER_L_PER_PERSON_PER_DAY / 1000

    # Prepare columns to fill
    if final_columns is None:
        # Default: all expected_mgL_values keys
        final_columns = ['YEAR', 'FLOYR'] + list(expected_mgL_values.keys())
    else:
        # Ensure 'YEAR' and 'FLOYR' are present
        if 'YEAR' not in final_columns:
            final_columns = ['YEAR'] + final_columns
        if 'FLOYR' not in final_columns:
            final_columns = ['YEAR', 'FLOYR'] + [col for col in final_columns if col not in ('YEAR', 'FLOYR')]

    # Print union/intersection for debug
    expected_vars = set(expected_mgL_values.keys())
    requested_vars = set(final_columns)
    print("Columns in expected_mgL_values:", expected_vars)
    print("Requested final columns:", requested_vars)
    print("Intersection (will be filled):", expected_vars & requested_vars)
    print("Missing in expected_mgL_values (will be filled with 0):", requested_vars - expected_vars - {'YEAR', 'FLOYR', 'POPULATION'})
    print("Extra in expected_mgL_values (not requested):", expected_vars - requested_vars)

    # Fill columns
    for col in final_columns:
        if col in ('YEAR', 'FLOYR', 'POPULATION'):
            continue
        elif col in expected_mgL_values:
            mgL = expected_mgL_values[col]
            df_out[col] = df_out['POPULATION'].apply(lambda p: round(mgL_to_kg_day(mgL, p), 6))
        else:
            df_out[col] = 0

    # Reorder columns
    df_out = df_out[[c for c in final_columns if c in df_out.columns] + [c for c in df_out.columns if c not in final_columns]]

    return df_out

## Get number of years our swat model simulates (from file.cio)

In [18]:
def getModelParameter(prameter:str,parameterfile:str)->int|str|float|None:
        with open(parameterfile,"r") as f:
            for line in f.readlines():
                if(line.find(prameter)!=-1):
                   return line.partition("|")[0].strip()

def getSimulatedPeriod(swatiofile: str) -> tuple[int, int]:
    skip_year = int(getModelParameter("NYSKIP", swatiofile))
    sim_year = int(getModelParameter("NBYR", swatiofile))
    start_year = int(getModelParameter("IYR", swatiofile))
    start_sim_year = start_year + skip_year
    end_sim_year = start_sim_year + sim_year - 1
    return start_sim_year, end_sim_year

start_year, end_year = getSimulatedPeriod(cio_file)
print(f"start_year = {start_year} \nend_year   = {end_year}")

start_year = 1981 
end_year   = 2020


## Constructing SWAT ready Tables

In [19]:
def build_swat_ready_tables(input_df, expected_mgL_values, start_year: int, end_year: int, id_column='GRIDCODE', swat_columns_order=None):
    """
    For an input DataFrame (wide format: ID + year columns),
    generate a dictionary of SWAT-ready DataFrames per ID, limited to a specific simulation period.
    
    Parameters:
    - input_df: DataFrame with one row per unit (e.g., subbasin) and columns: ID + year cols
    - expected_mgL_values: dictionary of variable: mg/L values
    - start_year: first year to include (inclusive)
    - end_year: last year to include (inclusive)
    - id_column: the column name identifying each unit (default: 'GRIDCODE')
    
    Returns:
    - dict { id_value: DataFrame with yearly SWAT variables }
    """
    # Filter only year columns within the simulation period
    years = [int(col) for col in input_df.columns if col.isdigit() and start_year <= int(col) <= end_year]
    
    swat_ready_dataframes = {}

    for idx, row in input_df.iterrows():
        id_value = row[id_column]
        swat_ready_dataframes[id_value] = build_point_load_timeseries_dataframes(row, expected_mgL_values, years, swat_columns_order)
    
    return swat_ready_dataframes


In [20]:
swat_columns_order = [
    "YEAR", "FLOYR", "SEDYR", "ORGNYR", "ORGPYR", "NO3YR", "NH3YR", "NO2YR",
    "MINPYR", "CBODYR", "DISOXYR", "CHLAYR", "SOLPSTYR", "SRBPSTYR",
    "BACTPYR", "BACTLPYR", "CMTL1YR", "CMTL2YR", "CMTL3YR"
]



swat_ready_dataframes = build_swat_ready_tables(df_1970_to_2025, expected_mgL_values=expected_mgL_values, start_year=start_year, end_year=end_year, id_column='identifier', swat_columns_order=swat_columns_order)
swat_ready_dataframes[17]

NameError: name 'df_1970_to_2025' is not defined

### Save to swat ready .dat files

In [21]:
import os

def write_recyear_files(
    swat_ready_dataframes: dict,
    output_folder: str,
    start_year: int,
    end_year: int,
    swat_columns_order: list,
    df_for_metadata=None
):
    """
    Write each SWAT-ready dataframe to a .dat file following a strict column format:
    - 1 space + 4 right-aligned chars for YEAR column.
    - 1 space + 16 chars for other columns (right-aligned).
    - Floats are adjusted dynamically to fit exactly 16 chars.
    """

    os.makedirs(output_folder, exist_ok=True)

    def format_float_16(value: float) -> str:
        """
        Format a float to exactly 16 characters
        Adjust precision dynamically so the total string length is always 16.
        """
        # Start with scientific notation and trim/expand
        s = f"{value:.10E}"  # start with high precision
        if len(s) > 16:
            # Reduce precision if too long
            for p in range(9, -1, -1):
                s = f"{value:.{p}E}"
                if len(s) <= 16:
                    break
        else:
            # If shorter, pad left
            s = s.rjust(16)
        return s

    for id_value, df in swat_ready_dataframes.items():
        present_columns = [col for col in swat_columns_order if col in df.columns]
        df_filtered = df[present_columns]

        filename = f"rcyr_{id_value}.dat"
        filepath = os.path.join(output_folder, filename)

        # Drainage area
        drainage_area = None
        if df_for_metadata is not None and "GRIDCODE" in df_for_metadata.columns and "Area" in df_for_metadata.columns:
            match = df_for_metadata[df_for_metadata["GRIDCODE"] == id_value]
            if not match.empty:
                area_ha = match.iloc[0]["Area"]
                drainage_area = area_ha / 100.0  # ha to km²

        with open(filepath, 'w') as f:
            # Metadata
            area_str = f"{drainage_area:.3f}" if drainage_area else "0.000"
            f.write(f" TITLE LINE 1 - Subbasin ID {id_value} | Simulation Years: {start_year}-{end_year} | DRAINAGE_AREA (km²): {area_str}\n")
            f.write(" TITLE LINE 2 - Source: TrabajoFM model\n")
            f.write(" TITLE LINE 3 - Units: kg/day\n")
            f.write(f" TITLE LINE 4 - Period: {start_year}-{end_year}\n")
            f.write(" TITLE LINE 5 - \n")

            # Header line
            header_line = f"{'YEAR':>5}"  # 1 space + 4 chars for YEAR
            for col in present_columns:
                if col != 'YEAR':
                    header_line += f"{col:>17}"  # 1 space + 16 chars
            f.write(header_line + "\n")

            # Data lines
            for _, row in df_filtered.iterrows():
                line = f"{int(row.iloc[0]):>5}"  # YEAR
                for v in row.iloc[1:]:
                    line += " " + format_float_16(float(v))
                f.write(line + "\n")

        print(f"✅ File saved: {filepath}")


In [22]:
write_recyear_files(swat_ready_dataframes, output_dir, start_year, end_year, swat_columns_order, df_for_metadata=df)

NameError: name 'swat_ready_dataframes' is not defined

# Archived:

## ArcPy verion

In [ ]:
import arcpy
import os
import glob
from arcpy.sa import ZonalStatisticsAsTable, CellStatistics




import matplotlib.pyplot as plt
import numpy as np

def visualize_raster_and_zones(raster_path, zones_fc, zone_field, year, output_dir):
    """Visualize the raster and sub-basin zones side by side."""
    from arcpy import Raster, da
    import matplotlib.pyplot as plt

    # Convert raster to NumPy array and extract metadata
    raster = Raster(raster_path)
    arr = arcpy.RasterToNumPyArray(raster)
    extent = raster.extent
    lower_left_x = extent.XMin
    lower_left_y = extent.YMin
    cell_size_x = raster.meanCellWidth
    cell_size_y = raster.meanCellHeight

    # Create extent for imshow
    extent = [lower_left_x,
              lower_left_x + arr.shape[1] * cell_size_x,
              lower_left_y,
              lower_left_y + arr.shape[0] * cell_size_y]

    # Extract zone boundaries as polygons (outline only)
    tmp_layer = "tmp_zones_layer"
    arcpy.MakeFeatureLayer_management(zones_fc, tmp_layer)
    shapes = []
    with da.SearchCursor(tmp_layer, ["SHAPE@"]) as cursor:
        for row in cursor:
            shapes.append(row[0])

    # Create matplotlib plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(arr, extent=extent, origin='upper', cmap='viridis')
    for shape in shapes:
        part = shape.getPart(0)
        coords = [(pt.X, pt.Y) for pt in part if pt]
        if coords:
            xs, ys = zip(*coords)
            ax.plot(xs, ys, color='red', linewidth=1)

    ax.set_title(f"Clipped Raster with Zone Boundaries ({year})")
    ax.set_xlabel("Easting")
    ax.set_ylabel("Northing")

    out_png = os.path.join(output_dir, f"raster_zones_arc_{year}.png")
    fig.tight_layout()
    fig.savefig(out_png)
    plt.close()
    print(f"📷 Visualization saved: {out_png}")


import arcpy
import os
import glob
from arcpy.sa import ZonalStatisticsAsTable
import matplotlib.pyplot as plt
import numpy as np

def subbasinPopulationAggregationToGPKG(raster_folder, sub_basin_fc, zone_field, output_gpkg, use_cache=True):
    arcpy.env.overwriteOutput = True
    arcpy.env.workspace = "in_memory"
    arcpy.CheckOutExtension("spatial")

    working_fc = "in_memory/basin_working_copy"
    if arcpy.Exists(working_fc):
        arcpy.Delete_management(working_fc)
    arcpy.management.CopyFeatures(sub_basin_fc, working_fc)

    tif_files = glob.glob(os.path.join(raster_folder, "hipgdac_es_100_*.tif"))
    cache_folder = os.path.join(raster_folder, "temp_cache")
    os.makedirs(cache_folder, exist_ok=True)

    for raster_path in tif_files:
        year = os.path.splitext(os.path.basename(raster_path))[0].split("_")[-1]
        field_name = f"pop_{year}"
        print(f"\n=== Processing raster: {raster_path} (year: {year}) ===")

        projected_raster = os.path.join(cache_folder, f"proj_{year}.tif")
        if not use_cache or not arcpy.Exists(projected_raster):
            print(f"→ Projecting raster...")
            arcpy.management.ProjectRaster(
                in_raster=raster_path,
                out_raster=projected_raster,
                out_coor_system=arcpy.SpatialReference(25830)
            )
        else:
            print(f"✅ Using cached projected raster: {projected_raster}")

        # Print global raster stats
        desc = arcpy.Describe(projected_raster)
        print(f"→ Projected raster cell size: {desc.meanCellWidth} x {desc.meanCellHeight}")
        for prop in ["MINIMUM", "MAXIMUM", "MEAN"]:
            val = arcpy.GetRasterProperties_management(projected_raster, prop)
            print(f"   {prop}: {val.getOutput(0)}")

        clipped_raster = os.path.join(cache_folder, f"clip_{year}.tif")
        if not use_cache or not arcpy.Exists(clipped_raster):
            print(f"→ Clipping raster...")
            arcpy.management.Clip(
                in_raster=projected_raster,
                rectangle="438912.5 4123462.5 470812.5 4158362.5",
                out_raster=clipped_raster,
                in_template_dataset=sub_basin_fc,
                clipping_geometry="ClippingGeometry"
            )
        else:
            print(f"✅ Using cached clipped raster: {clipped_raster}")

        print(f"→ Clipped raster stats:")
        for prop in ["MINIMUM", "MAXIMUM", "MEAN"]:
            val = arcpy.GetRasterProperties_management(clipped_raster, prop)
            print(f"   {prop}: {val.getOutput(0)}")

        # Visualization
        visualize_raster_and_zones(clipped_raster, sub_basin_fc, zone_field, year, raster_folder)

        # Zonal statistics
        zonal_table = os.path.join(cache_folder, f"zonal_{year}.dbf")
        if not use_cache or not arcpy.Exists(zonal_table):
            print(f"→ Running ZonalStatisticsAsTable...")
            ZonalStatisticsAsTable(
                in_zone_data=sub_basin_fc,
                zone_field=zone_field,
                in_value_raster=clipped_raster,
                out_table=zonal_table,
                ignore_nodata="DATA",
                statistics_type="SUM"
            )
        else:
            print(f"✅ Using cached zonal table: {zonal_table}")

        # Inspect values
        print("→ Raw zonal table values (SUM + COUNT):")
        with arcpy.da.SearchCursor(zonal_table, [zone_field, "SUM", "COUNT"]) as cursor:
            for row in cursor:
                print(f"   Zone {row[0]} → SUM: {row[1]}, COUNT: {row[2]}, AVG: {float(row[1])/row[2] if row[2] else 'NA'}")


        count_field = f"px_{year}"  # e.g. px_1970

        # Create new fields
        arcpy.management.AddField(zonal_table, field_name, "DOUBLE")
        arcpy.management.AddField(zonal_table, count_field, "LONG")

        # Copy over data
        arcpy.management.CalculateField(zonal_table, field_name, "!SUM!", "PYTHON3")
        arcpy.management.CalculateField(zonal_table, count_field, "!COUNT!", "PYTHON3")

        # Optionally delete original fields
        try:
            arcpy.management.DeleteField(zonal_table, ["SUM", "COUNT"])
        except Exception as e:
            print(f"⚠️ Warning: Could not delete original SUM/COUNT fields — {e}")


        # Join to working_fc
        arcpy.management.JoinField(
            in_data=working_fc,
            in_field=zone_field,
            join_table=zonal_table,
            join_field=zone_field,
            fields=[field_name, f"pixels_included_{year}"]
        )

    # Output final feature class
    output_layer_name = "population_by_subbasin"
    temp_output_fc = os.path.join(arcpy.env.scratchGDB, output_layer_name)
    arcpy.management.CopyFeatures(in_features=working_fc, out_feature_class=temp_output_fc)
    arcpy.conversion.FeatureClassToFeatureClass(
        in_features=temp_output_fc,
        out_path=output_gpkg,
        out_name=output_layer_name
    )

    arcpy.management.Delete("in_memory")
    print("\n✅ Completed zonal statistics with caching support.")
    return os.path.join(output_gpkg, output_layer_name)

raster_folder = r"..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\fgoerlich-HIPGDAC-ES-cd11f21\HIPGDAC-ES\test_copy"
sub_basin_fc = r"..\..\Genil GEO_INFO_POOL\SWaT outputs\Cubillas\cubillas_first_run\shapes\Sub_basin.shp"
output_folder = r"..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations"
output_gpkg = output_folder + r"\cubillas_population.gpkg"



result_fc = subbasinPopulationAggregationToGPKG(
    raster_folder,
    sub_basin_fc,
    zone_field="GRIDCODE",
    output_gpkg =output_gpkg,
    use_cache=False  # switch to False to force rerun
)


# Extract attributes to pandas DataFrame
import pandas as pd
fields = [f.name for f in arcpy.ListFields(result_fc) if f.type not in ('OID', 'Geometry')]
with arcpy.da.SearchCursor(result_fc, fields) as cursor:
    data = list(cursor)
df = pd.DataFrame(data, columns=fields)

# Export to CSV
csv_output = output_gpkg.replace(".gpkg", ".csv")
df.to_csv(csv_output, sep=';', index=False)
print(f"Exported to CSV: {csv_output}")

In [ ]:

raster_folder = r"..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\fgoerlich-HIPGDAC-ES-cd11f21\HIPGDAC-ES\test_copy"
sub_basin_fc = r"..\..\Genil GEO_INFO_POOL\SWaT outputs\Cubillas\cubillas_first_run\shapes\Sub_basin.shp"
output_folder = r"..\..\Genil GEO_INFO_POOL\Input Data\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations"
output_gpkg = output_folder + r"\cubillas_population.gpkg"



result_fc = subbasinPopulationAggregationToGPKG(
    raster_folder,
    sub_basin_fc,
    zone_field="GRIDCODE",
    output_gpkg =output_gpkg,
    use_cache=False  # switch to False to force rerun
)


# Extract attributes to pandas DataFrame
import pandas as pd
fields = [f.name for f in arcpy.ListFields(result_fc) if f.type not in ('OID', 'Geometry')]
with arcpy.da.SearchCursor(result_fc, fields) as cursor:
    data = list(cursor)
df = pd.DataFrame(data, columns=fields)

# Export to CSV
csv_output = output_gpkg.replace(".gpkg", ".csv")
df.to_csv(csv_output, sep=';', index=False)
print(f"Exported to CSV: {csv_output}")

## save swat ready dataframes to .csv

In [ ]:
""" 

import os

# save the DataFrames to Excel files

output_folder = 'C:\\Users\\Usuario\\OneDrive - UNIVERSIDAD DE HUELVA\\Granada\\TrabajoFM\\Genil GEO_INFO_POOL\\Input Data\\Population data\\Basin Aggregations\\Cubillas population loads\\test'

os.makedirs(output_folder, exist_ok=True)

for name, df in swat_ready_dataframes.items():
    safe_name = str(name).replace(' ', '_').replace('/', '_')  # clean filename
    file_path = os.path.join(output_folder, f'{safe_name}.xlsx')  # or .csv
    df.to_excel(file_path, index=False)  # or df.to_csv()


 """

## master dataframe of timeseries of all point loads for all subbasins together:

In [ ]:

# master dataframe of timeseries of all point loads for all subbasins together:

# Get all DataFrames into a list
dfs = list(swat_ready_dataframes.values())

# Get the name of the first (non-numeric) column
id_column_name = dfs[0].columns[0]
id_column_values = dfs[0][id_column_name]

# Initialize summed DataFrame (excluding identifier column)
summed_df = dfs[0].drop(columns=[id_column_name]).copy()

# Add the rest
for df in dfs[1:]:
    summed_df += df.drop(columns=[id_column_name])

# Optionally reattach the identifier if you want (but it may not make sense when summing)
summed_df.insert(0, id_column_name, id_column_values)



# Manual test to ensure the summed DataFrame is correct
population_sum_1970 = 0
for df in dfs:
    population_sum_1970 += df[df['YEAR'] == 1970].iloc[0, 1]
print(f"Total population in 1970 across all subbasins: {population_sum_1970}")

# Result: summed_df contains the matrix sum of all numeric columns
summed_df

summed_df.to_csv(output_folder+r"\summbed_cubillas_point_loads_arcgis_workflow.csv", index=False)
